# NSL-KDD Attack Type Segregation Analysis

This notebook computes:
- Segregation by exact attack label (`label`)
- Segregation by broad category (Normal / DoS / Probe / R2L / U2R)
- Optional CSV exports

Update the input file path in the next cell and run all cells.

In [3]:
from pathlib import Path
import pandas as pd

# === Configuration ===
# Change this path to any compatible CSV file.
INPUT_FILE = Path('data/train/KDDTrain+_20Percent.csv')

# Set to True if you want output CSV files.
EXPORT_RESULTS = True
OUTPUT_DIR = Path('data/train/analysis_output')

INPUT_FILE

WindowsPath('data/train/KDDTrain+_20Percent.csv')

In [4]:
# Load data and basic checks
df = pd.read_csv(INPUT_FILE)
print(f'Loaded rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')

required_col = 'label'
if required_col not in df.columns:
    raise ValueError(f"Expected column '{required_col}' not found. Available columns: {list(df.columns)}")

df.head(3)

Loaded rows: 25,192
Columns: 43


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,neptune,19


In [5]:
# Segregation by exact label
total = len(df)
label_dist = (
    df['label']
    .value_counts(dropna=False)
    .rename_axis('label')
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
label_dist['percent'] = (label_dist['count'] / total * 100).round(4)

print(f'Total samples: {total:,}')
print(f'Unique labels: {label_dist.shape[0]}')
label_dist

Total samples: 25,192
Unique labels: 22


,label,count,percent
0,normal,13449,53.3860
1,neptune,8282,32.8755
2,ipsweep,710,2.8184
3,satan,691,2.7429
4,portsweep,587,2.3301
5,smurf,529,2.0999
6,nmap,301,1.1948
7,back,196,0.7780
8,teardrop,188,0.7463
9,warezclient,181,0.7185


In [6]:
# Segregation by broad attack category
category_map = {
    'normal': 'Normal',
    # DoS
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS',
    # Probe
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    # R2L
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    # U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'rootkit': 'U2R'
}

df['attack_category'] = df['label'].map(category_map).fillna('Unknown')

category_dist = (
    df['attack_category']
    .value_counts(dropna=False)
    .rename_axis('category')
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
category_dist['percent'] = (category_dist['count'] / total * 100).round(4)

category_dist

,category,count,percent
0,Normal,13449,53.3860
1,DoS,9234,36.6545
2,Probe,2289,9.0862
3,R2L,209,0.8296
4,U2R,11,0.0437


In [7]:
# Normal vs Attack summary
normal_count = int((df['label'] == 'normal').sum())
attack_count = total - normal_count

summary = pd.DataFrame({
    'metric': ['Total', 'Normal', 'Attack'],
    'count': [total, normal_count, attack_count],
    'percent': [100.0, round(normal_count / total * 100, 4), round(attack_count / total * 100, 4)]
})

summary

,metric,count,percent
0,Total,25192,100.000
1,Normal,13449,53.386
2,Attack,11743,46.614


In [8]:
# Optional exports
if EXPORT_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    label_path = OUTPUT_DIR / f'{INPUT_FILE.stem}_label_distribution.csv'
    cat_path = OUTPUT_DIR / f'{INPUT_FILE.stem}_category_distribution.csv'
    enriched_path = OUTPUT_DIR / f'{INPUT_FILE.stem}_with_attack_category.csv'

    label_dist.to_csv(label_path, index=False)
    category_dist.to_csv(cat_path, index=False)
    df.to_csv(enriched_path, index=False)

    print('Saved:')
    print(f'- {label_path}')
    print(f'- {cat_path}')
    print(f'- {enriched_path}')
else:
    print('EXPORT_RESULTS is False; no files were written.')

Saved:
- data\train\analysis_output\KDDTrain+_20Percent_label_distribution.csv
- data\train\analysis_output\KDDTrain+_20Percent_category_distribution.csv
- data\train\analysis_output\KDDTrain+_20Percent_with_attack_category.csv
